# Проверки `c_nazn` в `ods.scd1_z_main_docum`

Цели:

1. Посмотреть все уникальные варианты `c_nazn` за первую неделю июня.
2. Выделить варианты `c_nazn`, которые могут относиться к эквайрингу.

Период задается параметрами ниже (по умолчанию: с `2026-06-01` по `2026-06-07` включительно).

In [ ]:
import os
import pandas as pd
from rail_connectors.connection import connect

print('Imports loaded')

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры периода: первая неделя июня
week_start = '2026-06-01'
week_end_exclusive = '2026-06-08'  # c 01 по 07 июня включительно

# Параметры исполнения
mem_limit = '8g'
preview_limit = 500
load_all_unique = False
load_all_ekv = False

table_name = 'ods.scd1_z_main_docum'

# Жесткие правила нормализации для группировки
# 1) "по договору" НЕ режем, чтобы не терять важные эквайринговые формулировки.
# 2) Межбанковское вознаграждение UnionPay CPD ... -> канон.
# 3) Комиссия за обслуживание карт (service fee) UnionPay ... -> канон.
# 4) Зачисление(я) по массиву -> канон "зачисления пенсии".
# 5) Перевод за услугу Билайн/Т2 через ПАО "Промсвязьбанк" с датой операции -> каноны.
# 6) "Пособия, компенсации, меры социальной поддержки%" -> "меры соц поддержки".
#
# Для UnionPay используем маркер union|юнион|pay|пэй, чтобы пережить
# смешанную кириллицу/латиницу в реальных данных.
union_token_condition_sql = (
    "(nazn_norm like '%union%' or nazn_norm like '%юнион%' "
    "or nazn_norm like '%pay%' or nazn_norm like '%пэй%')"
)

unionpay_interbank_condition_sql = (
    f"(nazn_norm like '%межбанковск%' "
    f"and nazn_norm like '%вознагражден%' "
    f"and nazn_norm like '%cpd%' "
    f"and {union_token_condition_sql})"
)
unionpay_interbank_canonical = 'Межбанковское вознаграждение UnionPay'

unionpay_service_fee_condition_sql = (
    f"({union_token_condition_sql} "
    f"and (nazn_norm like '%service fee%' or nazn_norm like '%servicefee%') "
    f"and nazn_norm like '%обслуживан%' "
    f"and nazn_norm like '%платежн%' "
    f"and nazn_norm like '%карт%')"
)
unionpay_service_fee_canonical = 'Комиссия, уплаченная за обслуживание платежных карт (service fee) UnionPay'

pension_mass_condition_sql = (
    "(nazn_norm like '%зачислен% по массив%' "
    "or nazn_norm like '%для зачисления по массив%')"
)
pension_mass_canonical = 'зачисления пенсии'

beeline_psb_transfer_condition_sql = (
    "(nazn_norm like '%перевод денежных средств за услугу%' "
    "and nazn_norm like '%в пользу билайн%' "
    "and nazn_norm like '%промсвязьбанк%' "
    "and nazn_norm like '%дата операции%')"
)
beeline_psb_transfer_canonical = 'Перевод денежных средств за услугу "Билайн" в пользу Билайн через ПАО "Промсвязьбанк"'

t2_psb_transfer_condition_sql = (
    "(nazn_norm like '%перевод денежных средств за услугу%' "
    "and (nazn_norm like '%в пользу t2%' or nazn_norm like '%в пользу т2%') "
    "and nazn_norm like '%промсвязьбанк%' "
    "and nazn_norm like '%дата операции%')"
)
t2_psb_transfer_canonical = 'Перевод денежных средств за услугу "T2" в пользу T2 через ПАО "Промсвязьбанк"'

social_support_condition_sql = (
    "(nazn_norm like '%пособи%' "
    "and nazn_norm like '%компенсац%' "
    "and nazn_norm like '%социальн%' "
    "and nazn_norm like '%поддержк%')"
)
social_support_canonical = 'меры соц поддержки'

rule_id_pension_mass = 'rule_pension_mass'
rule_id_unionpay_interbank = 'rule_unionpay_interbank'
rule_id_unionpay_service_fee = 'rule_unionpay_service_fee'
rule_id_beeline_psb_transfer = 'rule_beeline_psb_transfer'
rule_id_t2_psb_transfer = 'rule_t2_psb_transfer'
rule_id_social_support = 'rule_social_support'
rule_id_raw = 'rule_raw'

print(f'Period: [{week_start}, {week_end_exclusive})')
print(f'table={table_name}')

In [ ]:
# Подключение к Impala (как в 01_07_acq_dash.ipynb)
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connection initialized')

## 1) Все уникальные варианты `c_nazn` за первую неделю июня

In [ ]:
sql_unique_stats = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when {pension_mass_condition_sql}
                then '{pension_mass_canonical}'
            when {social_support_condition_sql}
                then '{social_support_canonical}'
            when {unionpay_service_fee_condition_sql}
                then '{unionpay_service_fee_canonical}'
            when {unionpay_interbank_condition_sql}
                then '{unionpay_interbank_canonical}'
            when {beeline_psb_transfer_condition_sql}
                then '{beeline_psb_transfer_canonical}'
            when {t2_psb_transfer_condition_sql}
                then '{t2_psb_transfer_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped,
        case
            when {pension_mass_condition_sql}
                then '{rule_id_pension_mass}'
            when {social_support_condition_sql}
                then '{rule_id_social_support}'
            when {unionpay_service_fee_condition_sql}
                then '{rule_id_unionpay_service_fee}'
            when {unionpay_interbank_condition_sql}
                then '{rule_id_unionpay_interbank}'
            when {beeline_psb_transfer_condition_sql}
                then '{rule_id_beeline_psb_transfer}'
            when {t2_psb_transfer_condition_sql}
                then '{rule_id_t2_psb_transfer}'
            else '{rule_id_raw}'
        end as rule_id
    from scoped
)
select
    count(*) as total_rows,
    count(distinct c_nazn_raw) as unique_raw_c_nazn_count,
    count(distinct c_nazn_grouped) as unique_grouped_c_nazn_count,
    sum(case when rule_id <> '{rule_id_raw}' then 1 else 0 end) as rows_with_applied_rules,
    count(distinct case when rule_id <> '{rule_id_raw}' then c_nazn_grouped end) as unique_groups_from_rules
from normalized
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    unique_stats_df = imp.fetch(sql_unique_stats)

unique_stats_df

In [ ]:
sql_unique_variants_base = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when {pension_mass_condition_sql}
                then '{pension_mass_canonical}'
            when {social_support_condition_sql}
                then '{social_support_canonical}'
            when {unionpay_service_fee_condition_sql}
                then '{unionpay_service_fee_canonical}'
            when {unionpay_interbank_condition_sql}
                then '{unionpay_interbank_canonical}'
            when {beeline_psb_transfer_condition_sql}
                then '{beeline_psb_transfer_canonical}'
            when {t2_psb_transfer_condition_sql}
                then '{t2_psb_transfer_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped,
        case
            when {pension_mass_condition_sql}
                then '{rule_id_pension_mass}'
            when {social_support_condition_sql}
                then '{rule_id_social_support}'
            when {unionpay_service_fee_condition_sql}
                then '{rule_id_unionpay_service_fee}'
            when {unionpay_interbank_condition_sql}
                then '{rule_id_unionpay_interbank}'
            when {beeline_psb_transfer_condition_sql}
                then '{rule_id_beeline_psb_transfer}'
            when {t2_psb_transfer_condition_sql}
                then '{rule_id_t2_psb_transfer}'
            else '{rule_id_raw}'
        end as rule_id
    from scoped
)
select
    c_nazn_grouped as c_nazn,
    rule_id,
    count(*) as cnt,
    count(distinct c_nazn_raw) as raw_variants_collapsed
from normalized
group by c_nazn_grouped, rule_id
order by cnt desc
"""

sql_unique_variants = (
    sql_unique_variants_base
    if load_all_unique
    else sql_unique_variants_base + f'\nlimit {preview_limit}'
)

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    unique_variants_df = imp.fetch(sql_unique_variants)

print(f'Rows loaded: {len(unique_variants_df):,}')
unique_variants_df.head(50)

In [ ]:
# Диагностика правил: токены + покрытие rule_id
sql_rule_diagnostics = f"""
with scoped as (
    select
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        nazn_norm,
        case
            when {pension_mass_condition_sql}
                then '{rule_id_pension_mass}'
            when {social_support_condition_sql}
                then '{rule_id_social_support}'
            when {unionpay_service_fee_condition_sql}
                then '{rule_id_unionpay_service_fee}'
            when {unionpay_interbank_condition_sql}
                then '{rule_id_unionpay_interbank}'
            when {beeline_psb_transfer_condition_sql}
                then '{rule_id_beeline_psb_transfer}'
            when {t2_psb_transfer_condition_sql}
                then '{rule_id_t2_psb_transfer}'
            else '{rule_id_raw}'
        end as rule_id
    from scoped
)
select
    sum(case when {union_token_condition_sql} then 1 else 0 end) as union_token_hits,
    sum(case when nazn_norm like '%межбанковск%' then 1 else 0 end) as token_mezhbank_hits,
    sum(case when nazn_norm like '%вознагражден%' then 1 else 0 end) as token_voznagr_hits,
    sum(case when {unionpay_interbank_condition_sql} then 1 else 0 end) as unionpay_interbank_condition_hits,
    sum(case when nazn_norm like '%service fee%' or nazn_norm like '%servicefee%' then 1 else 0 end) as token_service_fee_hits,
    sum(case when {unionpay_service_fee_condition_sql} then 1 else 0 end) as unionpay_service_fee_condition_hits,
    sum(case when {pension_mass_condition_sql} then 1 else 0 end) as pension_mass_condition_hits,
    sum(case when {beeline_psb_transfer_condition_sql} then 1 else 0 end) as beeline_psb_transfer_condition_hits,
    sum(case when {t2_psb_transfer_condition_sql} then 1 else 0 end) as t2_psb_transfer_condition_hits,
    sum(case when {social_support_condition_sql} then 1 else 0 end) as social_support_condition_hits,
    sum(case when rule_id = '{rule_id_unionpay_interbank}' then 1 else 0 end) as rule_unionpay_interbank_rows,
    sum(case when rule_id = '{rule_id_unionpay_service_fee}' then 1 else 0 end) as rule_unionpay_service_fee_rows,
    sum(case when rule_id = '{rule_id_pension_mass}' then 1 else 0 end) as rule_pension_mass_rows,
    sum(case when rule_id = '{rule_id_beeline_psb_transfer}' then 1 else 0 end) as rule_beeline_psb_transfer_rows,
    sum(case when rule_id = '{rule_id_t2_psb_transfer}' then 1 else 0 end) as rule_t2_psb_transfer_rows,
    sum(case when rule_id = '{rule_id_social_support}' then 1 else 0 end) as rule_social_support_rows,
    sum(case when rule_id = '{rule_id_raw}' then 1 else 0 end) as rule_raw_rows
from normalized
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    rule_diagnostics_df = imp.fetch(sql_rule_diagnostics)

rule_diagnostics_df

## 2) Варианты `c_nazn`, которые могут относиться к эквайрингу

In [ ]:
# Ловим слова от корня "эквайр" (эквайринг, эквайринга, эквайринговый и т.д.)
ekv_pattern = r'(^|[^а-яa-z0-9])(эквайр[а-я]*)([^а-яa-z0-9]|$)'

sql_ekv_stats = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when {pension_mass_condition_sql}
                then '{pension_mass_canonical}'
            when {social_support_condition_sql}
                then '{social_support_canonical}'
            when {unionpay_service_fee_condition_sql}
                then '{unionpay_service_fee_canonical}'
            when {unionpay_interbank_condition_sql}
                then '{unionpay_interbank_canonical}'
            when {beeline_psb_transfer_condition_sql}
                then '{beeline_psb_transfer_canonical}'
            when {t2_psb_transfer_condition_sql}
                then '{t2_psb_transfer_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped,
        case
            when {pension_mass_condition_sql}
                then '{rule_id_pension_mass}'
            when {social_support_condition_sql}
                then '{rule_id_social_support}'
            when {unionpay_service_fee_condition_sql}
                then '{rule_id_unionpay_service_fee}'
            when {unionpay_interbank_condition_sql}
                then '{rule_id_unionpay_interbank}'
            when {beeline_psb_transfer_condition_sql}
                then '{rule_id_beeline_psb_transfer}'
            when {t2_psb_transfer_condition_sql}
                then '{rule_id_t2_psb_transfer}'
            else '{rule_id_raw}'
        end as rule_id
    from scoped
)
select
    count(*) as scoped_rows,
    sum(case when nazn_norm rlike '{ekv_pattern}' then 1 else 0 end) as matched_rows,
    count(distinct case when nazn_norm rlike '{ekv_pattern}' then c_nazn_grouped end) as matched_unique_grouped,
    sum(case when nazn_norm rlike '{ekv_pattern}' and rule_id <> '{rule_id_raw}' then 1 else 0 end) as matched_rows_with_rules
from normalized
"""

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    ekv_stats_df = imp.fetch(sql_ekv_stats)

ekv_stats_df

In [ ]:
sql_ekv_variants_base = f"""
with scoped as (
    select
        coalesce(c_nazn, '') as c_nazn_raw,
        trim(regexp_replace(regexp_replace(lower(coalesce(c_nazn, '')), 'ё', 'е'), '[[:space:]]+', ' ')) as nazn_norm
    from {table_name}
    where cast(c_date_prov as date) >= date '{week_start}'
      and cast(c_date_prov as date) < date '{week_end_exclusive}'
), normalized as (
    select
        c_nazn_raw,
        nazn_norm,
        case
            when {pension_mass_condition_sql}
                then '{pension_mass_canonical}'
            when {social_support_condition_sql}
                then '{social_support_canonical}'
            when {unionpay_service_fee_condition_sql}
                then '{unionpay_service_fee_canonical}'
            when {unionpay_interbank_condition_sql}
                then '{unionpay_interbank_canonical}'
            when {beeline_psb_transfer_condition_sql}
                then '{beeline_psb_transfer_canonical}'
            when {t2_psb_transfer_condition_sql}
                then '{t2_psb_transfer_canonical}'
            else c_nazn_raw
        end as c_nazn_grouped,
        case
            when {pension_mass_condition_sql}
                then '{rule_id_pension_mass}'
            when {social_support_condition_sql}
                then '{rule_id_social_support}'
            when {unionpay_service_fee_condition_sql}
                then '{rule_id_unionpay_service_fee}'
            when {unionpay_interbank_condition_sql}
                then '{rule_id_unionpay_interbank}'
            when {beeline_psb_transfer_condition_sql}
                then '{rule_id_beeline_psb_transfer}'
            when {t2_psb_transfer_condition_sql}
                then '{rule_id_t2_psb_transfer}'
            else '{rule_id_raw}'
        end as rule_id
    from scoped
)
select
    c_nazn_grouped as c_nazn,
    rule_id,
    count(*) as cnt,
    count(distinct c_nazn_raw) as raw_variants_collapsed
from normalized
where nazn_norm rlike '{ekv_pattern}'
group by c_nazn_grouped, rule_id
order by cnt desc
"""

sql_ekv_variants = (
    sql_ekv_variants_base
    if load_all_ekv
    else sql_ekv_variants_base + f'\nlimit {preview_limit}'
)

with imp:
    imp.execute(f'set MEM_LIMIT={mem_limit}')
    ekv_variants_df = imp.fetch(sql_ekv_variants)

print(f'Rows loaded: {len(ekv_variants_df):,}')
ekv_variants_df.head(100)

In [ ]:
# Необязательно: сохранить результаты в CSV
save_to_csv = False
unique_out_path = './c_nazn_unique_first_week_june.csv'
ekv_out_path = './c_nazn_ekv_first_week_june.csv'

if save_to_csv:
    unique_variants_df.to_csv(unique_out_path, index=False)
    ekv_variants_df.to_csv(ekv_out_path, index=False)
    print(f'Saved: {unique_out_path}')
    print(f'Saved: {ekv_out_path}')
else:
    print('save_to_csv=False, nothing was written.')